# ChronoMind: Google Drive cache setup in Colab

Run the cells from top to bottom. The notebook clones the `sarthak` branch, mounts your Google Drive, downloads only the small train annotation file and the video for one sample, then verifies the dataset tensor. Your Google account is used only when Colab asks you to authorize the Drive mount; no token needs to be added to GitHub.

The dataset stays on Hugging Face as the source. Drive stores only files this notebook requests and the frame cache. The local `/content/chronomind_tmp` folder is temporary.

In [ ]:
import subprocess
import sys
from pathlib import Path

repo_url = "https://github.com/sarthak-shastri/ChronoMind.git"
repo_dir = Path("/content/ChronoMind")

if not repo_dir.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", "sarthak", repo_url, str(repo_dir)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only", "origin", "sarthak"], check=True)

import os
os.chdir(repo_dir)
print("Project folder:", Path.cwd())

In [ ]:
# Colab already provides PyTorch and NumPy. Install the Hub and video/runtime packages
# without replacing Colab's PyTorch build.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "huggingface_hub", "opencv-python-headless", "psutil", "PyYAML"],
    check=True,
)
import torch
import cv2
print("PyTorch:", torch.__version__, "CUDA available:", torch.cuda.is_available())
print("OpenCV:", cv2.__version__)

In [ ]:
from google.colab import drive

drive_available = False
try:
    drive.mount("/content/drive")
    drive_available = True
except Exception as exc:
    print("Drive could not be mounted:", exc)
    print("Continuing with temporary Colab storage. Cached files will be lost when this runtime ends.")


In [ ]:
from preprocessing.storage import resolve_storage_paths

storage = resolve_storage_paths({
    "storage": {
        "drive_root": "/content/drive/MyDrive/ChronoMind_reva_cache" if drive_available else None,
        "local_cache_root": "/content/chronomind_tmp",
    }
})
print("Persistent Drive cache:", "enabled" if drive_available else "unavailable; using temporary storage")
for name, path in storage.items():
    print(f"{name}: {path}")

In [ ]:
import json
from preprocessing.hf_access import fetch_annotation_file

manifest = json.loads(Path("manifests/reva_manifest.json").read_text(encoding="utf-8"))
repo_id = manifest["repo_id"]
revision = manifest["revision"]
annotation_file = fetch_annotation_file(
    repo_id,
    "train.json",
    revision=revision,
    cache_dir=storage["hf_cache_dir"],
)
print("Pinned revision:", revision)
print("Training annotations cached at:", annotation_file)

In [ ]:
from preprocessing import ReVADataset

# Start with one record so only its referenced MP4 is fetched.
dataset = ReVADataset(
    annotation_file,
    dataset_repo_id=repo_id,
    revision=revision,
    hf_cache_dir=storage["hf_cache_dir"],
    cache_dir=storage["frame_cache_dir"],
    num_frames=8,
    sampling="uniform",
    sample_ids=["train_000001"],
)
sample = dataset[0]
print("Sample:", sample["sample_id"])
print("Answer:", sample["answer"])
print("Tensor:", tuple(sample["video_frames"].shape), sample["video_frames"].dtype)
print("Frame cache misses:", dataset.cache.misses, "hits:", dataset.cache.hits)

In [ ]:
# Read the same sample again to verify the reusable sampled-frame cache.
_ = dataset[0]
print("After repeat — frame cache misses:", dataset.cache.misses, "hits:", dataset.cache.hits)

## What this verified

This checks the pinned annotation download, one-video lazy download, 8-frame tensor output, and repeat-read frame cache. If Drive mounted, the cache persists in your Drive. If Drive authorization fails, the notebook continues with temporary Colab storage, which is deleted when the runtime ends. It deliberately does not fetch the complete dataset.